In [1]:
import requests
import os
import time
import json
import logging
import pandas as pd
from datetime import datetime, timezone

# ── API Configuration ──────────────────────────────────────────────────────
# These are pre-set as environment variables inside Docker
# We don't hardcode credentials — that's a security bad practice
API_BASE_URL = os.environ.get("API_BASE_URL", "http://mock-api:8000")
API_USERNAME  = os.environ.get("API_USERNAME",  "candidate")
API_PASSWORD  = os.environ.get("API_PASSWORD",  "blue-owls-2026")

# ── Output paths ───────────────────────────────────────────────────────────
OUTPUT_DIR    = "/home/jovyan/work/output/bronze"
MANIFEST_PATH = "/home/jovyan/work/output/manifest.json"
DATE_FROM     = "2018-07-01"
PAGE_SIZE     = 1000

# ── Logging setup ──────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s"
)
log = logging.getLogger(__name__)

# Create output folder if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ Config loaded successfully")
print(f"   API URL  : {API_BASE_URL}")
print(f"   Output   : {OUTPUT_DIR}")

✅ Config loaded successfully
   API URL  : http://mock-api:8000
   Output   : /home/jovyan/work/output/bronze


In [2]:
# ── Token Management ───────────────────────────────────────────────────────
# The API requires a Bearer token to access data
# Tokens expire after 15 minutes, so we need to refresh them automatically

_token = None  # We store the token here so we don't fetch it repeatedly

def get_token():
    """Fetch a fresh token from the API."""
    global _token
    resp = requests.post(
        f"{API_BASE_URL}/api/v1/auth/token",
        json={"username": API_USERNAME, "password": API_PASSWORD},
        timeout=10
    )
    resp.raise_for_status()
    _token = resp.json()["access_token"]
    log.info("✅ Token fetched successfully.")
    return _token

def auth_headers():
    """Return headers with the current token. Fetch one if we don't have it."""
    if _token is None:
        get_token()
    return {"Authorization": f"Bearer {_token}"}

# Test it immediately
get_token()
print("✅ Authentication working")

2026-03-28 20:04:46,211 INFO ✅ Token fetched successfully.


✅ Authentication working


In [9]:
# ── Resilient GET with Retry Logic ─────────────────────────────────────────
def api_get(url, params=None, max_retries=10):
    """
    Makes a GET request with automatic retry and token refresh.
    
    - 401 = token expired → get a new token and retry immediately
    - 429 = too many requests → wait and retry with exponential backoff
    - 500 = server error → wait and retry with exponential backoff
    """
    for attempt in range(max_retries):
        time.sleep(0.5)  # small base delay on every attempt
        try:
            resp = requests.get(
                url,
                headers=auth_headers(),
                params=params,
                timeout=15
            )

            if resp.status_code == 401:
                log.warning("Token expired — refreshing and retrying...")
                get_token()
                continue

            if resp.status_code == 429:
                wait = 2 ** attempt
                log.warning(f"Rate limited (429) — waiting {wait}s before retry {attempt+1}/{max_retries}")
                time.sleep(wait)
                continue

            if resp.status_code == 500:
                wait = 2 ** attempt
                log.warning(f"Server error (500) — waiting {wait}s before retry {attempt+1}/{max_retries}")
                time.sleep(wait)
                continue

            resp.raise_for_status()
            return resp.json()

        except requests.RequestException as e:
            wait = 2 ** attempt
            log.error(f"Request failed: {e} — waiting {wait}s before retry {attempt+1}/{max_retries}")
            time.sleep(wait)

    raise RuntimeError(f"❌ All {max_retries} attempts failed for: {url}")

print("✅ Resilient API function defined")

✅ Resilient API function defined


In [4]:
# ── Manifest ───────────────────────────────────────────────────────────────
# The manifest is a simple JSON file that tracks which pages we have
# already successfully ingested from each endpoint.
#
# Why? If the pipeline crashes halfway through and we restart it,
# we don't want to re-download pages we already have.
# This is what makes our pipeline IDEMPOTENT.
#
# Example manifest.json:
# {
#   "orders": [1, 2, 3, 4, 5],
#   "customers": [1, 2]
# }

def load_manifest():
    """Load the manifest file. Return empty dict if it doesn't exist yet."""
    if os.path.exists(MANIFEST_PATH):
        with open(MANIFEST_PATH) as f:
            return json.load(f)
    return {}

def save_manifest(manifest):
    """Save the manifest file after each successful page ingestion."""
    with open(MANIFEST_PATH, "w") as f:
        json.dump(manifest, f, indent=2)

def is_page_done(endpoint, page):
    """Check if a specific page for an endpoint was already ingested."""
    manifest = load_manifest()
    return page in manifest.get(endpoint, [])

def mark_page_done(endpoint, page):
    """Mark a page as successfully ingested in the manifest."""
    manifest = load_manifest()
    manifest.setdefault(endpoint, [])
    if page not in manifest[endpoint]:
        manifest[endpoint].append(page)
    save_manifest(manifest)

# Test it
mark_page_done("test_endpoint", 1)
manifest = load_manifest()
print("✅ Manifest working")
print(f"   Manifest contents: {manifest}")

# Clean up test entry
manifest.pop("test_endpoint", None)
save_manifest(manifest)
print("   Test entry cleaned up")

✅ Manifest working
   Manifest contents: {'test_endpoint': [1]}
   Test entry cleaned up


In [5]:
# ── Ingestion Function ─────────────────────────────────────────────────────
# This function pulls ALL pages from ONE endpoint and saves to a CSV file.
# It uses everything we built above:
#   - api_get() for resilient calls
#   - manifest to skip already-ingested pages
#   - quarantine file for malformed records

def ingest_endpoint(endpoint, date_filter=False):
    """
    Pull all pages from an endpoint and save as bronze CSV.
    
    endpoint    : e.g. "orders", "customers", "products"
    date_filter : if True, adds date_from=2018-07-01 to filter old records
    """
    url = f"{API_BASE_URL}/api/v1/data/{endpoint}"
    page = 1
    all_records = []
    ingested_at = datetime.now(timezone.utc).isoformat()

    log.info(f"Starting ingestion for: {endpoint}")

    while True:
        # Skip this page if we already ingested it
        if is_page_done(endpoint, page):
            log.info(f"  [{endpoint}] page {page} already done — skipping")
            page += 1
            # Check if there are more pages by trying the next one
            params = {"page": page, "page_size": PAGE_SIZE}
            if date_filter:
                params["date_from"] = DATE_FROM
            data = api_get(url, params)
            records = data if isinstance(data, list) else data.get("data", [])
            if not records:
                break
            continue

        # Build request parameters
        params = {"page": page, "page_size": PAGE_SIZE}
        if date_filter:
            params["date_from"] = DATE_FROM

        log.info(f"  [{endpoint}] fetching page {page}...")
        data = api_get(url, params)

        # Handle both list and dict response formats
        if isinstance(data, list):
            records = data
        elif isinstance(data, dict):
            records = data.get("data", [])
        else:
            records = []

        # No more records — we've reached the last page
        if not records:
            log.info(f"  [{endpoint}] no more records at page {page} — done")
            break

        # ── Validate each record ───────────────────────────────────────────
        # Good records go into all_records
        # Bad records go into a quarantine file for investigation
        valid_count = 0
        quarantine_count = 0

        for r in records:
            if not isinstance(r, dict) or not r:
                # Malformed record — save to quarantine
                qpath = f"{OUTPUT_DIR}/{endpoint}_quarantine.jsonl"
                with open(qpath, "a") as qf:
                    qf.write(json.dumps({"raw": str(r), "reason": "not a dict or empty", "page": page}) + "\n")
                quarantine_count += 1
            else:
                # Valid record — add metadata columns and keep
                all_records.append({
                    **r,
                    "_ingested_at": ingested_at,
                    "_source_endpoint": endpoint
                })
                valid_count += 1

        if quarantine_count > 0:
            log.warning(f"  [{endpoint}] page {page}: {valid_count} valid, {quarantine_count} quarantined")
        else:
            log.info(f"  [{endpoint}] page {page}: {valid_count} records")

        # Mark this page as done in the manifest
        mark_page_done(endpoint, page)

        page += 1
        time.sleep(0.3)  # small delay to be polite to the API

    # ── Save to CSV ────────────────────────────────────────────────────────
    if all_records:
        df_new = pd.DataFrame(all_records)
        out_path = f"{OUTPUT_DIR}/{endpoint}.csv"

        if os.path.exists(out_path):
            # File exists — append and deduplicate
            df_existing = pd.read_csv(out_path)
            df_combined = pd.concat([df_existing, df_new], ignore_index=True)
            df_final = df_combined.drop_duplicates()
            df_final.to_csv(out_path, index=False)
            log.info(f"  [{endpoint}] ✅ updated CSV — {len(df_final)} total rows")
        else:
            # First time — just save
            df_new.to_csv(out_path, index=False)
            log.info(f"  [{endpoint}] ✅ saved CSV — {len(df_new)} rows")
    else:
        log.info(f"  [{endpoint}] nothing new to write")

    return all_records

print("✅ Ingestion function defined")

✅ Ingestion function defined


In [10]:
# ── Run Ingestion for All Endpoints ───────────────────────────────────────
# orders and order_items support date filtering (date_from parameter)
# the rest don't support it so we fetch everything and filter later

DATE_ENDPOINTS = ["orders", "order_items"]
ALL_ENDPOINTS  = ["orders", "order_items", "customers", "products", "sellers", "payments"]

# Authenticate once upfront
get_token()

# Run ingestion for each endpoint one by one
results = {}
for endpoint in ALL_ENDPOINTS:
    log.info(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    log.info(f"Processing: {endpoint}")
    records = ingest_endpoint(endpoint, date_filter=(endpoint in DATE_ENDPOINTS))
    results[endpoint] = len(records)

# ── Summary ────────────────────────────────────────────────────────────────
print("\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("✅ INGESTION COMPLETE — Summary:")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
for endpoint, count in results.items():
    print(f"   {endpoint:<20} {count:>6} records")

# ── Verify files were created ──────────────────────────────────────────────
print("\n📁 Bronze files created:")
for f in os.listdir(OUTPUT_DIR):
    if f.endswith(".csv"):
        filepath = f"{OUTPUT_DIR}/{f}"
        df = pd.read_csv(filepath)
        print(f"   {f:<35} {len(df):>6} rows, {len(df.columns)} columns")

2026-03-28 20:32:38,580 INFO ✅ Token fetched successfully.
2026-03-28 20:32:38,582 INFO ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
2026-03-28 20:32:38,583 INFO Processing: orders
2026-03-28 20:32:38,585 INFO Starting ingestion for: orders
2026-03-28 20:32:38,588 INFO   [orders] page 1 already done — skipping
2026-03-28 20:32:39,304 INFO   [orders] page 2 already done — skipping
2026-03-28 20:32:39,868 INFO   [orders] page 3 already done — skipping
2026-03-28 20:32:40,429 INFO   [orders] page 4 already done — skipping
2026-03-28 20:32:40,989 INFO   [orders] page 5 already done — skipping
2026-03-28 20:32:41,548 INFO   [orders] page 6 already done — skipping
2026-03-28 20:32:42,057 WARNING Server error (500) — waiting 1s before retry 1/10
2026-03-28 20:32:43,568 WARNING Rate limited (429) — waiting 2s before retry 2/10
2026-03-28 20:32:46,137 INFO   [orders] page 7 already done — skipping
2026-03-28 20:32:46,705 INFO   [orders] page 8 already done — skipping
2026-03-28 20:32:47,256 INFO   


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ INGESTION COMPLETE — Summary:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   orders                    0 records
   order_items               0 records
   customers             53441 records
   products              32951 records
   sellers                3095 records
   payments             103886 records

📁 Bronze files created:
   order_items.csv                      15590 rows, 9 columns
   orders.csv                           12824 rows, 10 columns
   products.csv                         32951 rows, 11 columns
   payments.csv                        103886 rows, 7 columns
   sellers.csv                           3095 rows, 6 columns
   customers.csv                        53441 rows, 7 columns
